# Clinical, Imaging, Laboratory, and Metagenomic Data Exploration with `mlcroissant`
This notebook demonstrates how to load, review, and process a dataset using the `mlcroissant` library, referencing all entities by their `@id` value as per the Croissant schema.

### Dataset Source
The dataset is defined via the Croissant schema:
[https://sen.science/doi/10.71728/senscience.s88k-ggjn/fair2.json](https://sen.science/doi/10.71728/senscience.s88k-ggjn/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.s88k-ggjn/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

# Also print the dataset @id for reference
print(f"Dataset @id: {metadata.id}")

## 2. Data Overview
List available record sets, fields, and columns by their `@id`.

In [ ]:
# Get the recordSets
record_sets = metadata.record_sets

print("Record Sets Available:")
for rs in record_sets:
    print(f"- {rs.id}: {rs.name}")

# For each record set, show the fields and columns
for rs in record_sets:
    print(f"\nRecord Set @id: {rs.id} ({rs.name})")
    print(" Fields:")
    for field in rs.fields:
        print(f"   - {field.id}: {field.name} (Type: {field.data_type})")
    if hasattr(rs, "columns"):
        print(" Columns:")
        for col in rs.columns:
            print(f"   - {col.id}: {col.name}")

## 3. Data Extraction
Load records from a selected record set referenced by its `@id`. All records, fields, and columns will always be referenced by `@id` values provided by the Croissant schema.

In [ ]:
# Build a list of record set IDs (using `@id`)
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
print(f"Record Set @ids: {record_set_ids}")

# Load records from each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nSample records for {record_set_id}: {df.head(2)}")

# Choose the first record set as example
selected_record_set_id = record_set_ids[0]
print(f"\nColumns in {selected_record_set_id}: {dataframes[selected_record_set_id].columns.tolist()}")
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All references to fields use their `@id`.

In [ ]:
# Select a numeric field for EDA
# List available fields with numeric types
numeric_fields = []
for field in record_sets[0].fields:
    if field.data_type in ["Integer", "Float", "Number"]:
        numeric_fields.append(field.id)
print(f"Numeric field candidates in {selected_record_set_id}: {numeric_fields}")

# Choose the first numeric field
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Selected numeric field for filtering: {numeric_field_id}")
    df = dataframes[selected_record_set_id]
    # Remove rows where numeric field is missing
    df = df.dropna(subset=[numeric_field_id])
    threshold = df[numeric_field_id].mean()
    # Filter records where the value is greater than the threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by a categorical field (choose first non-numeric)
    group_field_id = None
    for field in record_sets[0].fields:
        if field.id != numeric_field_id and field.data_type not in ["Integer", "Float", "Number"]:
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric fields found for EDA in the selected record set.")

## 5. Visualization
Visualize distributions and relationships between fields, using their `@id`.

In [ ]:
# Visualize the distribution of the numeric field
if numeric_fields:
    plt.figure(figsize=(8, 4))
    filtered_df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id exists, show a bar chart
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df.set_index(group_field_id)[numeric_field_id].plot(kind="bar", figsize=(10, 5))
        plt.title(f"Mean {numeric_field_id} grouped by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.show()
else:
    print("No numeric fields found for visualization.")

## 6. Conclusion
This notebook provided an overview and initial exploration of the FAIR² clinical, imaging, laboratory, and metagenomic dataset. Entities were referenced via their Croissant `@id` values to ensure semantic clarity and reproducibility. Basic filtering and normalization steps were showcased for one record set. For further exploration, consult record set and field `@id`s as shown above.

Key findings include:
- Listing available record sets and their structure (`@id`, fields, columns).
- Using `mlcroissant` to load and filter numeric fields, normalize values, and summarize group differences.
- Visualizing distributions using field `@id`s for reproducible analysis.

Next steps might include deeper cleaning, joining record sets, or advanced analytics depending on the experimental or clinical research goals.